# Standardized patient-level benchmark: AlexNet, VGG16, DenseNet121 and EfficientNetB0

This notebook rebuilds the comparison using the same experimental protocol as the proposed MobileNetV1--Fourier-INR experiment:

- input size: 128 x 128;
- six classes;
- batch size: 4 for cross-validation and final training;
- maximum 50 epochs;
- early-stopping patience: 15, monitoring validation loss;
- Adam learning rate: 1e-4;
- balanced class weights;
- no data augmentation;
- seed: 42;
- the exact locked patient folds generated by the MobileNet experiment;
- the same architecture is used for ten-fold and final-model evaluation;
- the untouched patient-independent test cohort is used only for final evaluation.

Important: AlexNet is initialized randomly because TensorFlow does not provide built-in ImageNet weights for AlexNet. VGG16, DenseNet121 and EfficientNetB0 use ImageNet-pretrained backbones. Model-specific ImageNet preprocessing is applied inside each transfer-learning model.


In [1]:
# =====================================================================
# 1. IMPORTS AND LOCKED CONFIGURATION
# =====================================================================
import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"

import gc
import json
import time
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
import tensorflow as tf

from tensorflow.keras import backend as K
from tensorflow.keras import layers, models, regularizers
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, precision_score, recall_score,
    f1_score, cohen_kappa_score, matthews_corrcoef,
    classification_report, confusion_matrix,
)
from sklearn.utils.class_weight import compute_class_weight
from scipy.stats import friedmanchisquare, wilcoxon

IMG_SIZE = 128
NUM_CLASSES = 6
BATCH_SIZE = 4
EPOCHS = 50
PATIENCE = 15
LEARNING_RATE = 1e-4
L2_VALUE = 1e-4
DROPOUT_RATE = 0.5
SEED = 42
N_SPLITS = 10
INFERENCE_REPEATS = 3

def holm_correction(p_values, alpha=0.05):
    """Dependency-free Holm--Bonferroni correction."""
    p_values = np.asarray(p_values, dtype=float)
    if p_values.ndim != 1:
        raise ValueError("p_values must be one-dimensional.")
    if np.any(~np.isfinite(p_values)) or np.any((p_values < 0) | (p_values > 1)):
        raise ValueError("Every p-value must be finite and lie in [0, 1].")

    number_of_tests = len(p_values)
    if number_of_tests == 0:
        return np.array([], dtype=bool), np.array([], dtype=float)

    sorted_indices = np.argsort(p_values)
    sorted_p_values = p_values[sorted_indices]
    sorted_adjusted = np.empty(number_of_tests, dtype=float)
    running_maximum = 0.0

    for rank, p_value in enumerate(sorted_p_values, start=1):
        corrected_value = (number_of_tests - rank + 1) * p_value
        running_maximum = max(running_maximum, corrected_value)
        sorted_adjusted[rank - 1] = min(running_maximum, 1.0)

    adjusted_p_values = np.empty(number_of_tests, dtype=float)
    adjusted_p_values[sorted_indices] = sorted_adjusted
    reject = adjusted_p_values <= alpha
    return reject, adjusted_p_values

MODEL_NAMES = ["AlexNet", "VGG16", "DenseNet121", "EfficientNetB0"]
CLASS_NAMES = ["Abdomen", "Brain", "Femur", "Maternal Cervix", "Thorax", "Other"]

MOBILENET_RESULTS_DIR = Path(
    r"D:\imagecondition revision\MobileNetV1_ABCD_PatientLevel_10Fold_6Class_Results"
)
MANIFEST_FILE = MOBILENET_RESULTS_DIR / "Dataset_Manifest_with_Patient_IDs.csv"
LOCKED_FOLDS_FILE = MOBILENET_RESULTS_DIR / "Shared_Patient_Fold_Assignments.csv"

# New folder: existing batch-16/augmented outputs are never overwritten.
RESULTS_DIR = Path(
    r"D:\imagecondition revision\CNN_Benchmark_Batch4_NoAug_PatientLevel_10Fold_6Class_Results"
)
FOLD_DIR = RESULTS_DIR / "folds"
FINAL_DIR = RESULTS_DIR / "final_model"
for directory in (RESULTS_DIR, FOLD_DIR, FINAL_DIR):
    directory.mkdir(parents=True, exist_ok=True)

np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)

print("TensorFlow:", tf.__version__)
print("GPU:", tf.config.list_physical_devices("GPU"))
print("Batch size:", BATCH_SIZE)
print("Augmentation: NONE")
print("Results:", RESULTS_DIR)


TensorFlow: 2.10.0
GPU: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
Batch size: 4
Augmentation: NONE
Results: D:\imagecondition revision\CNN_Benchmark_Batch4_NoAug_PatientLevel_10Fold_6Class_Results


In [2]:
# =====================================================================
# 2. LOAD THE MOBILENET MANIFEST AND EXACT LOCKED PATIENT FOLDS
# =====================================================================
for required_file in (MANIFEST_FILE, LOCKED_FOLDS_FILE):
    if not required_file.exists():
        raise FileNotFoundError(f"Required MobileNet experiment file not found:\n{required_file}")

manifest = pd.read_csv(MANIFEST_FILE)
required_manifest_columns = {"split", "path", "label", "class_name", "patient_id"}
missing = required_manifest_columns - set(manifest.columns)
if missing:
    raise ValueError(f"Manifest columns missing: {sorted(missing)}")

manifest["split"] = manifest["split"].astype(str).str.strip().str.lower()
manifest["patient_id"] = manifest["patient_id"].astype(str)
manifest["label"] = manifest["label"].astype(int)

train_df = manifest[manifest.split == "train"].reset_index(drop=True)
val_df = manifest[manifest.split == "val"].reset_index(drop=True)
test_df = manifest[manifest.split == "test"].reset_index(drop=True)
development_df = pd.concat([train_df, val_df], ignore_index=True)

def patient_set(frame):
    return set(frame.patient_id.astype(str))

assert not (patient_set(train_df) & patient_set(val_df))
assert not (patient_set(train_df) & patient_set(test_df))
assert not (patient_set(val_df) & patient_set(test_df))

assignments = pd.read_csv(LOCKED_FOLDS_FILE)
required_assignment_columns = {"Fold", "Role", "Patient ID"}
missing = required_assignment_columns - set(assignments.columns)
if missing:
    raise ValueError(f"Locked-fold columns missing: {sorted(missing)}")
assignments["Patient ID"] = assignments["Patient ID"].astype(str)
assignments["Fold"] = assignments["Fold"].astype(int)

fold_plan = []
for fold in range(1, N_SPLITS + 1):
    fold_assignments = assignments[assignments.Fold == fold]
    role_patients = {
        role: set(fold_assignments.loc[fold_assignments.Role == role, "Patient ID"])
        for role in ("fit", "inner_validation", "outer_test")
    }
    fit_df = development_df[development_df.patient_id.isin(role_patients["fit"])].reset_index(drop=True)
    stop_df = development_df[development_df.patient_id.isin(role_patients["inner_validation"])].reset_index(drop=True)
    outer_df = development_df[development_df.patient_id.isin(role_patients["outer_test"])].reset_index(drop=True)

    if fit_df.empty or stop_df.empty or outer_df.empty:
        raise ValueError(f"Fold {fold} contains an empty role after applying locked assignments.")
    assert not (patient_set(fit_df) & patient_set(stop_df))
    assert not (patient_set(fit_df) & patient_set(outer_df))
    assert not (patient_set(stop_df) & patient_set(outer_df))
    fold_plan.append((fit_df, stop_df, outer_df))

print("Train/validation/test images:", len(train_df), len(val_df), len(test_df))
print("Train/validation/test patients:",
      train_df.patient_id.nunique(), val_df.patient_id.nunique(), test_df.patient_id.nunique())
print("Patient leakage: 0")
print("Locked folds loaded:", len(fold_plan))


Train/validation/test images: 10203 1062 1135
Train/validation/test patients: 1433 179 180
Patient leakage: 0
Locked folds loaded: 10


In [3]:
# =====================================================================
# 3. DATA PIPELINE — SHUFFLING ONLY; NO AUGMENTATION
# =====================================================================
AUTOTUNE = tf.data.AUTOTUNE

def decode_record(path, label):
    image = tf.io.read_file(path)
    image = tf.io.decode_image(image, channels=3, expand_animations=False)
    image.set_shape([None, None, 3])
    image = tf.image.resize(image, [IMG_SIZE, IMG_SIZE], method="bilinear")
    image = tf.cast(image, tf.float32) / 255.0
    target = tf.one_hot(tf.cast(label, tf.int32), NUM_CLASSES)
    return image, target

def make_dataset(frame, training=False):
    dataset = tf.data.Dataset.from_tensor_slices((
        frame.path.astype(str).to_numpy(),
        frame.label.astype(np.int32).to_numpy(),
    ))
    if training:
        # Shuffling changes sample order only and is not augmentation.
        dataset = dataset.shuffle(
            min(len(frame), 10000), seed=SEED, reshuffle_each_iteration=True
        )
    dataset = dataset.map(decode_record, num_parallel_calls=AUTOTUNE)
    dataset = dataset.batch(BATCH_SIZE, drop_remainder=False)
    return dataset.prefetch(AUTOTUNE)

def balanced_class_weights(labels):
    labels = np.asarray(labels, dtype=np.int32)
    classes = np.arange(NUM_CLASSES)
    if set(np.unique(labels)) != set(classes):
        raise ValueError("A training partition is missing one or more classes.")
    weights = compute_class_weight("balanced", classes=classes, y=labels)
    return {int(key): float(value) for key, value in zip(classes, weights)}

print("No random flip, rotation, brightness, zoom, crop, or other augmentation is present.")


No random flip, rotation, brightness, zoom, crop, or other augmentation is present.


In [4]:
# =====================================================================
# 4. ONE MODEL FACTORY USED FOR BOTH CROSS-VALIDATION AND FINAL TRAINING
# =====================================================================
def common_classifier_head(feature_map, prefix):
    x = layers.GlobalAveragePooling2D(name=f"{prefix}_gap")(feature_map)
    x = layers.Dense(
        256, activation="relu", kernel_regularizer=regularizers.l2(L2_VALUE),
        name=f"{prefix}_dense256",
    )(x)
    x = layers.BatchNormalization(name=f"{prefix}_bn")(x)
    x = layers.Dropout(DROPOUT_RATE, name=f"{prefix}_dropout")(x)
    return layers.Dense(NUM_CLASSES, activation="softmax", name="classification")(x)

def build_alexnet():
    inputs = layers.Input((IMG_SIZE, IMG_SIZE, 3), name="image_input")
    x = layers.Conv2D(96, 11, strides=4, padding="same", activation="relu",
                      kernel_regularizer=regularizers.l2(L2_VALUE))(inputs)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D(3, strides=2, padding="same")(x)
    x = layers.Conv2D(256, 5, padding="same", activation="relu",
                      kernel_regularizer=regularizers.l2(L2_VALUE))(x)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D(3, strides=2, padding="same")(x)
    x = layers.Conv2D(384, 3, padding="same", activation="relu",
                      kernel_regularizer=regularizers.l2(L2_VALUE))(x)
    x = layers.Conv2D(384, 3, padding="same", activation="relu",
                      kernel_regularizer=regularizers.l2(L2_VALUE))(x)
    x = layers.Conv2D(256, 3, padding="same", activation="relu",
                      kernel_regularizer=regularizers.l2(L2_VALUE))(x)
    x = layers.MaxPooling2D(3, strides=2, padding="same")(x)
    outputs = common_classifier_head(x, "alexnet")
    return models.Model(inputs, outputs, name="AlexNet")

def build_transfer_model(model_name):
    inputs = layers.Input((IMG_SIZE, IMG_SIZE, 3), name="image_input")
    pixel_255 = layers.Rescaling(255.0, name="restore_0_255_range")(inputs)

    if model_name == "VGG16":
        prepared = layers.Lambda(
            tf.keras.applications.vgg16.preprocess_input, name="vgg16_preprocess"
        )(pixel_255)
        backbone = tf.keras.applications.VGG16(
            include_top=False, weights="imagenet", input_shape=(IMG_SIZE, IMG_SIZE, 3)
        )
    elif model_name == "DenseNet121":
        prepared = layers.Lambda(
            tf.keras.applications.densenet.preprocess_input, name="densenet_preprocess"
        )(pixel_255)
        backbone = tf.keras.applications.DenseNet121(
            include_top=False, weights="imagenet", input_shape=(IMG_SIZE, IMG_SIZE, 3)
        )
    elif model_name == "EfficientNetB0":
        # TF 2.10 EfficientNet includes its own input rescaling and expects 0--255 input.
        prepared = pixel_255
        backbone = tf.keras.applications.EfficientNetB0(
            include_top=False, weights="imagenet", input_shape=(IMG_SIZE, IMG_SIZE, 3)
        )
    else:
        raise ValueError(f"Unsupported model: {model_name}")

    # The proposed MobileNet experiment freezes its pretrained backbone.
    # Apply the same rule to every ImageNet-pretrained comparison backbone.
    backbone.trainable = False
    feature_map = backbone(prepared, training=False)
    outputs = common_classifier_head(feature_map, model_name.lower())
    return models.Model(inputs, outputs, name=model_name)

def build_model(model_name):
    K.clear_session()
    gc.collect()
    model = build_alexnet() if model_name == "AlexNet" else build_transfer_model(model_name)
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=LEARNING_RATE),
        loss="categorical_crossentropy",
        metrics=["accuracy"],
    )
    return model

architecture_rows = []
for model_name in MODEL_NAMES:
    model = build_model(model_name)
    architecture_rows.append({
        "Model": model_name,
        "Parameters": model.count_params(),
        "Trainable Parameters": int(sum(K.count_params(w) for w in model.trainable_weights)),
        "Initialization": "Random" if model_name == "AlexNet" else "ImageNet",
    })
pd.DataFrame(architecture_rows).to_csv(
    RESULTS_DIR / "Architecture_Parameters.csv", index=False
)
print(pd.DataFrame(architecture_rows).to_string(index=False))
del model
K.clear_session(); gc.collect()


         Model  Parameters  Trainable Parameters Initialization
       AlexNet     3816966               3815750         Random
         VGG16    14848582                133382       ImageNet
   DenseNet121     7302470                264454       ImageNet
EfficientNetB0     4380073                329990       ImageNet


52191

In [5]:
# =====================================================================
# 5. METRICS, TIMING, AND SAFE RESUME HELPERS
# =====================================================================
class EpochTimer(tf.keras.callbacks.Callback):
    def __init__(self, model_name, run_name):
        super().__init__()
        self.model_name = model_name
        self.run_name = run_name
        self.rows = []
    def on_epoch_begin(self, epoch, logs=None):
        self.start = time.perf_counter()
    def on_epoch_end(self, epoch, logs=None):
        row = {
            "Model": self.model_name,
            "Run": self.run_name,
            "Epoch": epoch + 1,
            "Epoch Time (s)": time.perf_counter() - self.start,
        }
        row.update({key: float(value) for key, value in (logs or {}).items()})
        self.rows.append(row)

def metric_values(y_true, y_pred):
    return {
        "Accuracy": accuracy_score(y_true, y_pred),
        "Balanced Accuracy": balanced_accuracy_score(y_true, y_pred),
        "Macro Precision": precision_score(y_true, y_pred, average="macro", zero_division=0),
        "Macro Recall": recall_score(y_true, y_pred, average="macro", zero_division=0),
        "Macro F1": f1_score(y_true, y_pred, average="macro", zero_division=0),
        "Kappa": cohen_kappa_score(y_true, y_pred),
        "MCC": matthews_corrcoef(y_true, y_pred),
    }

def timed_predict(model, dataset, number_of_images):
    _ = model.predict(dataset.take(1), verbose=0)  # excluded warm-up
    elapsed = []
    probabilities = None
    for _ in range(INFERENCE_REPEATS):
        start = time.perf_counter()
        probabilities = model.predict(dataset, verbose=0)
        elapsed.append(time.perf_counter() - start)
    mean_total = float(np.mean(elapsed))
    return probabilities, mean_total, mean_total * 1000.0 / number_of_images

CV_RESULTS_FILE = RESULTS_DIR / "CNN_10Fold_All_Results.csv"
EPOCH_FILE = RESULTS_DIR / "CNN_All_Epoch_Timings.csv"

cv_rows = pd.read_csv(CV_RESULTS_FILE).to_dict("records") if CV_RESULTS_FILE.exists() else []
epoch_rows = pd.read_csv(EPOCH_FILE).to_dict("records") if EPOCH_FILE.exists() else []
completed = {(str(row["Model"]), int(row["Fold"])) for row in cv_rows}
print("Already completed model-fold pairs:", len(completed))


Already completed model-fold pairs: 28


In [6]:



















# =====================================================================
# 9. SAVE AN AUDITABLE RUN MANIFEST
# =====================================================================
run_manifest = {
    "created_at": datetime.now().isoformat(),
    "experiment": "Standardized CNN benchmark against MobileNetV1-Fourier-INR",
    "models": MODEL_NAMES,
    "image_size": IMG_SIZE,
    "classes": CLASS_NAMES,
    "batch_size_cv": BATCH_SIZE,
    "batch_size_final": BATCH_SIZE,
    "maximum_epochs": EPOCHS,
    "early_stopping_patience": PATIENCE,
    "learning_rate": LEARNING_RATE,
    "optimizer": "Adam",
    "loss": "categorical_crossentropy",
    "l2": L2_VALUE,
    "dropout": DROPOUT_RATE,
    "augmentation": False,
    "class_weights": "balanced",
    "seed": SEED,
    "patient_level_folds": True,
    "locked_fold_source": str(LOCKED_FOLDS_FILE),
    "dataset_manifest_source": str(MANIFEST_FILE),
    "untouched_test_used_for_model_selection": False,
    "initialization": {
        "AlexNet": "random",
        "VGG16": "ImageNet",
        "DenseNet121": "ImageNet",
        "EfficientNetB0": "ImageNet",
    },
}
with open(RESULTS_DIR / "Run_Manifest.json", "w", encoding="utf-8") as file:
    json.dump(run_manifest, file, indent=2)

print(json.dumps(run_manifest, indent=2))
print("\nAll standardized outputs saved to:", RESULTS_DIR)


{
  "created_at": "2026-08-23T17:10:23.896049",
  "experiment": "Standardized CNN benchmark against MobileNetV1-Fourier-INR",
  "models": [
    "AlexNet",
    "VGG16",
    "DenseNet121",
    "EfficientNetB0"
  ],
  "image_size": 128,
  "classes": [
    "Abdomen",
    "Brain",
    "Femur",
    "Maternal Cervix",
    "Thorax",
    "Other"
  ],
  "batch_size_cv": 4,
  "batch_size_final": 4,
  "maximum_epochs": 50,
  "early_stopping_patience": 15,
  "learning_rate": 0.0001,
  "optimizer": "Adam",
  "loss": "categorical_crossentropy",
  "l2": 0.0001,
  "dropout": 0.5,
  "augmentation": false,
  "class_weights": "balanced",
  "seed": 42,
  "patient_level_folds": true,
  "locked_fold_source": "D:\\imagecondition revision\\MobileNetV1_ABCD_PatientLevel_10Fold_6Class_Results\\Shared_Patient_Fold_Assignments.csv",
  "dataset_manifest_source": "D:\\imagecondition revision\\MobileNetV1_ABCD_PatientLevel_10Fold_6Class_Results\\Dataset_Manifest_with_Patient_IDs.csv",
  "untouched_test_used_for_model

In [7]:
# =====================================================================
# 9. SAVE AN AUDITABLE RUN MANIFEST
# =====================================================================
run_manifest = {
    "created_at": datetime.now().isoformat(),
    "experiment": "Standardized CNN benchmark against MobileNetV1-Fourier-INR",
    "models": MODEL_NAMES,
    "image_size": IMG_SIZE,
    "classes": CLASS_NAMES,
    "batch_size_cv": BATCH_SIZE,
    "batch_size_final": BATCH_SIZE,
    "maximum_epochs": EPOCHS,
    "early_stopping_patience": PATIENCE,
    "learning_rate": LEARNING_RATE,
    "optimizer": "Adam",
    "loss": "categorical_crossentropy",
    "l2": L2_VALUE,
    "dropout": DROPOUT_RATE,
    "augmentation": False,
    "class_weights": "balanced",
    "seed": SEED,
    "patient_level_folds": True,
    "locked_fold_source": str(LOCKED_FOLDS_FILE),
    "dataset_manifest_source": str(MANIFEST_FILE),
    "untouched_test_used_for_model_selection": False,
    "initialization": {
        "AlexNet": "random",
        "VGG16": "ImageNet",
        "DenseNet121": "ImageNet",
        "EfficientNetB0": "ImageNet",
    },
}
with open(RESULTS_DIR / "Run_Manifest.json", "w", encoding="utf-8") as file:
    json.dump(run_manifest, file, indent=2)

print(json.dumps(run_manifest, indent=2))
print("\nAll standardized outputs saved to:", RESULTS_DIR)


{
  "created_at": "2026-08-23T17:10:23.923307",
  "experiment": "Standardized CNN benchmark against MobileNetV1-Fourier-INR",
  "models": [
    "AlexNet",
    "VGG16",
    "DenseNet121",
    "EfficientNetB0"
  ],
  "image_size": 128,
  "classes": [
    "Abdomen",
    "Brain",
    "Femur",
    "Maternal Cervix",
    "Thorax",
    "Other"
  ],
  "batch_size_cv": 4,
  "batch_size_final": 4,
  "maximum_epochs": 50,
  "early_stopping_patience": 15,
  "learning_rate": 0.0001,
  "optimizer": "Adam",
  "loss": "categorical_crossentropy",
  "l2": 0.0001,
  "dropout": 0.5,
  "augmentation": false,
  "class_weights": "balanced",
  "seed": 42,
  "patient_level_folds": true,
  "locked_fold_source": "D:\\imagecondition revision\\MobileNetV1_ABCD_PatientLevel_10Fold_6Class_Results\\Shared_Patient_Fold_Assignments.csv",
  "dataset_manifest_source": "D:\\imagecondition revision\\MobileNetV1_ABCD_PatientLevel_10Fold_6Class_Results\\Dataset_Manifest_with_Patient_IDs.csv",
  "untouched_test_used_for_model